In [1]:
import numpy as np
import pandas as pd
import random

In [3]:
df = pd.read_csv('cities.csv')

df = df.rename(columns ={'0.046':'x', '4.006': 'y'})
df.head()


,x,y
0,-0.454,-0.706
1,0.181,5.769
2,0.739,4.741
3,-0.110,5.179
4,-0.235,0.271


In [6]:
centers = df[['x','y']].sample(3).to_numpy().tolist()
centers

[[4.261, 4.64], [4.662, 5.306], [5.172, 4.118]]

In [13]:
def assign_clusters(df, centers):
    clusters = {0:[], 1:[], 2:[]}
    
    for idx, row in df.iterrows():
        x,y = row['x'], row['y']

        distance = []
        for cx, cy in centers:
            dist = (cx-x)**2 + (cy-y)**2
            distance.append(dist)

        nearest = distance.index(min(distance))
        clusters[nearest].append((x, y))
    return clusters

In [16]:
clusters = assign_clusters(df,centers)

In [17]:
def update_centers_gd(clusters, lr=0.01, iterations =50):
    new_centers = []

    for k in clusters:
        points = clusters[k]

        if len(points) ==0:
            new_centers.append((0,0))
            continue

        cx = sum(p[0] for p in points)/len(points)
        cy = sum(p[1] for p in points)/len(points)

        for _ in range(iterations):
            grad_x = -2 * sum((x-cx) for x, y in points)
            grad_y = -2 * sum((y-cy) for x,y in points)

            cx = cx -lr*grad_x
            cy = cy -lr*grad_y
        
        new_centers.append((cx,cy))
    return new_centers

In [21]:
new_centers_gd = update_centers_gd(clusters)

In [ ]:
def update_centers_newton(clusters):
    new_centers = []

    for k in clusters:
        points = clusters[k]

        if len(points) == 0:
            new_centers.append((0, 0))
            continue

        cx = sum(p[0] for p in points) / len(points)
        cy = sum(p[1] for p in points) / len(points)

        new_centers.append((cx, cy))

    return new_centers

In [23]:
new_centers_newton = update_centers_newton(clusters)
new_centers_newton

[(np.float64(0.3093142857142857), np.float64(2.541371428571429)),
 (np.float64(4.989), np.float64(5.290899999999999)),
 (np.float64(5.21225), np.float64(4.33375))]

In [24]:
def compute_cost(clusters, centers):
    total_cost = 0

    for k in clusters:
        cx, cy = centers[k]

        for x, y in clusters[k]:
            dist = (x - cx)**2 + (y - cy)**2
            total_cost += dist

    return total_cost

In [28]:
# Gradient Descent
clusters_gd = assign_clusters(df, new_centers_gd)
cost_gd = compute_cost(clusters_gd, new_centers_gd)

# Newton Method
clusters_nt = assign_clusters(df, new_centers_newton)
cost_nt = compute_cost(clusters_nt, new_centers_newton)

print("Cost_GD:", cost_gd)
print("Cost_nt", cost_nt)

Cost_GD: 233.81469367765308
Cost_nt 233.81469367765308


In [30]:
def cluster_wise_cost(clusters, centers):
    costs = []

    for k in clusters:
        cx, cy = centers[k]
        cost = 0

        for x, y in clusters[k]:
            cost += (x - cx)**2 + (y - cy)**2

        costs.append(cost)

    return costs